# 직접 해 보기 2 — 파이썬 코드로 따라 해 보기

AI가 내놓은 결과만 보고 그것이 어떻게 나왔는지 모르면, 그 결과를 믿을 근거도 없다. AI도 실수를 하므로 결과는 점검해야 하고, 점검하려면 결과를 만든 파이썬 코드를 읽을 줄 알아야 한다.

이 노트북은 이 장에서 다룬 계산을 파이썬으로 직접 해 본다. 보고서의 네 모형(횡단면, 합동 OLS, 확률효과, 고정효과)을 다시 추정하고, 본문의 표를 차례로 만들고, 본문에서 숫자만 보았던 계산(자기 평균 빼기, $\theta$, 하우스만 통계량)을 손으로 해 본다. 코드는 될 수 있는 한 단순하게 썼다.

1. 데이터 읽기 — 이 장의 CSV를 읽는다(다른 수출국의 자료를 만드는 법도)
2. 자료 살펴보기 — 패널의 두 꼬리표, FTA를 맺은 나라와 해
3. 횡단면 — 2019년 한 해의 비교와 FTA가 없던 2003년(표 2-2)
4. 합동 OLS와 군집 표준오차 — 표 2-3, 군집을 반영하기 전과 후
5. 고정효과 — 자기 평균 빼기를 손으로, 나라 더미, 표 3-1, 두 차이를 나누는 계산
6. 확률효과 — 표 4-1, $\theta$를 손으로, 비교 방법에 따른 FTA의 계수(표 4-2)
7. 하우스만 검정 — 통계량을 손으로
8. 연도 효과 — 표 5-1과 해마다의 연도 효과
9. 더 해 보기: 발효 뒤 몇 해째인가 — 본문에서 다루지 않은 탐색

## 준비 — 패키지와 글꼴

`import`로 쓸 패키지를 불러온다. `as` 뒤의 이름은 줄임말이고, 이후 코드에서는 `np`, `pd`처럼 줄임말로 부른다.

이 장에는 새 패키지가 하나 있다. 패널 모형(합동 OLS, 고정효과, 확률효과)을 추정하는 `linearmodels`다. Anaconda와 Colab에 기본으로 들어 있지 않으므로 처음 한 번 설치한다. 내 컴퓨터에서는 Anaconda Prompt에서 `pip install linearmodels`를, Colab에서는 새 셀에서 `!pip install linearmodels`를 실행한다. 횡단면처럼 보통의 회귀분석은 다른 장과 같이 `statsmodels`로 한다.

matplotlib는 기본 글꼴에 한글이 없어서 그림의 한글이 네모로 깨진다. 그래서 한글 글꼴을 지정한다. 윈도우는 맑은 고딕(`Malgun Gothic`), 맥은 애플고딕(`AppleGothic`)을 쓴다. Colab에는 한글 글꼴이 없으므로 새 셀에서 `!apt-get -qq install fonts-nanum`을 실행하고, 런타임을 다시 시작한 뒤 `NanumGothic`을 지정한다.

In [ ]:
import numpy as np                       # 수치 계산
import pandas as pd                      # 표(데이터프레임) 다루기
import statsmodels.formula.api as smf    # 보통의 회귀분석(횡단면)
from linearmodels.panel import PooledOLS, PanelOLS, RandomEffects   # 패널 모형
from scipy.stats import chi2             # 하우스만 검정의 p값
import matplotlib.pyplot as plt          # 그림

# 그림의 한글 글꼴
# 맥은 'AppleGothic', Colab은 'NanumGothic'(설치 필요)
plt.rcParams['font.family'] = 'Malgun Gothic'   # 윈도우
plt.rcParams['axes.unicode_minus'] = False      # 음수 부호(-) 깨짐 방지

## 1. 데이터 읽기

이 장의 자료는 CEPII Gravity 데이터베이스(2022년 11월 판)에서 수출하는 나라를 한국으로 고정해 뽑은 것이다(1절의 "데이터"). 책 사이트의 이 장 첫 페이지에서 `korea-exports-panel-1995-2020.csv`로 내려받을 수 있다.

- 내 컴퓨터의 주피터: CSV를 이 노트북 파일과 같은 폴더에 두고, 아래 코드의 `CSV =` 줄에 파일 이름을 그대로 적는다.
- Colab: 왼쪽의 폴더 아이콘을 누르고 업로드 버튼으로 CSV를 올린 뒤 `CSV = '/content/파일이름.csv'`로 적는다.

이 CSV는 한국어 엑셀에서도 열리도록 UTF-8(BOM 포함)로 저장했다. `pd.read_csv`는 이것을 그대로 읽는다. 변수는 일곱 개다.

| 변수 | 뜻 |
|---|---|
| `연도` | 1995~2020 |
| `상대국코드`, `상대국` | 나라의 세 글자 코드(예: CHL)와 영문 이름 |
| `수출_천달러` | 한국이 그 나라에 그해 수출한 금액(천 달러, 명목) |
| `상대국GDP_천달러` | 그 나라의 그해 GDP(천 달러, 명목) |
| `거리_km` | 한국과 그 나라 사이의 거리(인구로 가중한 평균, km) |
| `FTA` | 그해 한국과 그 나라 사이에 FTA가 발효돼 있으면 1 |

In [ ]:
CSV = 'korea-exports-panel-1995-2020.csv'   # 자기 파일 이름으로 바꾼다

d = pd.read_csv(CSV)
print(len(d), '개 관측')

### 다른 수출국의 자료 만들기 (선택)

한국이 아닌 나라의 수출로 해 보려면 원자료에서 직접 뽑는다. CEPII 누리집(www.cepii.fr)의 Gravity 데이터베이스 쪽에서 `Gravity_V202211.csv`와 `Countries_V202211.csv`를 내려받는다. 출처를 밝히면 가공해서 다시 배포해도 되는 자료다. 원자료는 두 나라의 조합마다 해마다 한 행이라 파일이 크다(1GB가 넘는다). 그래서 한 번에 읽지 않고 `chunksize`로 200만 행씩 나눠 읽으며, 수출국(`iso3_o`)이 고른 나라인 행만 남긴다.

아래 셀은 `MAKE = True`로 바꿨을 때만 실행된다. `EXPORTER`에 수출국의 세 글자 코드(일본 `'JPN'`, 독일 `'DEU'`처럼)를 적는다. 뽑는 기준은 이 장의 자료와 같다. 1995~2020년, 수출액과 GDP와 거리와 FTA가 모두 있고 수출액이 0보다 큰 나라와 해만 남긴다.

In [ ]:
MAKE = False          # 다른 수출국의 자료를 만들 때만 True
EXPORTER = 'JPN'      # 수출국의 세 글자 코드

if MAKE:
    cols = ['year', 'iso3_o', 'iso3_d', 'distw_harmonic', 'gdp_d', 'fta_wto', 'tradeflow_comtrade_o']
    parts = [ch[ch.iso3_o == EXPORTER] for ch in pd.read_csv('Gravity_V202211.csv', usecols=cols, chunksize=2_000_000)]
    k = pd.concat(parts)
    k = k[(k.year >= 1995) & (k.year <= 2020)]
    k = k.dropna(subset=['tradeflow_comtrade_o', 'gdp_d', 'distw_harmonic', 'fta_wto'])
    k = k[k.tradeflow_comtrade_o > 0]
    # 나라 이름: 같은 코드에 옛 나라와 지금 나라가 함께 있으면 지금 나라(끝 연도가 비어 있음)를 쓴다
    ctry = pd.read_csv('Countries_V202211.csv')
    ctry['끝'] = ctry['last_year'].fillna(9999)
    names = ctry.sort_values('끝').drop_duplicates('iso3', keep='last').set_index('iso3')['country']
    out = pd.DataFrame({'연도': k.year, '상대국코드': k.iso3_d, '상대국': k.iso3_d.map(names),
                        '수출_천달러': k.tradeflow_comtrade_o.round(1), '상대국GDP_천달러': k.gdp_d.round(0),
                        '거리_km': k.distw_harmonic.round(1), 'FTA': k.fta_wto.astype(int)})
    out.sort_values(['상대국코드', '연도']).to_csv(f'{EXPORTER.lower()}-exports-panel-1995-2020.csv',
                                                  index=False, encoding='utf-8-sig')
    print('저장:', len(out), '개 관측')

## 2. 자료 살펴보기

회귀식에 넣을 로그 변수를 만든다. 수출과 GDP는 나라마다 수천 배씩 차이가 나서 로그를 취한다(로그-로그 회귀식의 계수는 탄력성으로 읽는다. 단순회귀분석 장 A5).

패널 데이터에서는 한 행에 꼬리표가 둘 붙는다. 어느 나라인지(`상대국코드`)와 어느 해인지(`연도`)다(2절). `linearmodels`는 이 두 꼬리표를 표의 색인(index)으로 받는다. `set_index(['상대국코드', '연도'])`가 그 일을 한다. 앞의 것이 대상(나라), 뒤의 것이 시점(해)이어야 한다.

FTA를 맺은 나라와 해도 확인한다. `groupby('상대국코드')`는 나라마다 묶고, 묶은 것마다 FTA가 처음 1이 된 해를 찾는다.

In [ ]:
d['로그수출'] = np.log(d['수출_천달러'])
d['로그GDP'] = np.log(d['상대국GDP_천달러'])
d['로그거리'] = np.log(d['거리_km'])

print('나라', d['상대국코드'].nunique(), '개,', d['연도'].min(), '~', d['연도'].max(), '년,', len(d), '개 관측')
print('26년 기록이 모두 있는 나라', (d.groupby('상대국코드').size() == 26).sum(), '개')

start = d[d['FTA'] == 1].groupby('상대국코드')['연도'].min()     # 나라마다 FTA가 처음 1이 된 해
print('FTA를 맺은 나라', len(start), '개')
print(start.value_counts().sort_index())                          # 해마다 몇 나라가 새로 맺었나

p = d.set_index(['상대국코드', '연도']).sort_index()              # 패널의 두 꼬리표: 나라, 해

## 3. 횡단면 — 2019년 한 해와 FTA가 없던 2003년

보고서 표의 맨 왼쪽 열은 2019년 한 해의 나라들만으로 추정한 횡단면이다. 회귀식은 다른 장과 같이 `smf.ols`로 쓴다. 표준오차는 보고서처럼 `cov_type='HC1'`(이분산에 강건한 표준오차)으로 계산한다.

종속변수가 로그이니 FTA 계수 $d$는 $e^d - 1$만큼, 즉 퍼센트로 바꿔 읽는다(다중회귀분석 장 B3). `np.expm1(d)`가 $e^d - 1$이다.

2절의 표 2-2는 같은 회귀식을 FTA가 하나도 없던 2003년에 적용했다. FTA 자리에는 나중에(2004~2020년) FTA를 맺게 되는 나라를 1로 넣는다. FTA의 효과가 있을 수 없는 해이니, 이 계수가 0에서 멀다면 한 해 비교의 FTA 계수에 원래 수출 규모의 차이가 섞여 있다는 신호다.

In [ ]:
YEAR = 2019    # 한 해 비교에 쓸 해

s = d[d['연도'] == YEAR]
cs = smf.ols('로그수출 ~ 로그GDP + 로그거리 + FTA', data=s).fit(cov_type='HC1')
print(YEAR, '년 횡단면 (나라', int(cs.nobs), '개)')
print(pd.DataFrame({'계수': cs.params, '표준오차': cs.bse, 'p값': cs.pvalues}).round(3))
print('FTA → 퍼센트', round(np.expm1(cs.params['FTA']) * 100), '%')

# 표 2-2: FTA가 하나도 없던 2003년, FTA 자리에 "나중에 FTA를 맺게 되는 나라"
d['나중FTA'] = d['상대국코드'].map(d.groupby('상대국코드')['FTA'].max())   # 언젠가 1이 되는 나라
s03 = d[d['연도'] == 2003]
cs03 = smf.ols('로그수출 ~ 로그GDP + 로그거리 + 나중FTA', data=s03).fit(cov_type='HC1')
print('\n2003년, 나중에 FTA를 맺게 되는 나라', int(s03['나중FTA'].sum()), '개')
print('계수', round(cs03.params['나중FTA'], 3), '표준오차', round(cs03.bse['나중FTA'], 3),
      'p값', round(cs03.pvalues['나중FTA'], 3), '→', round(np.expm1(cs03.params['나중FTA']) * 100), '%')

## 4. 합동 OLS와 군집 표준오차

합동 OLS는 26년 치를 나라 구분 없이 쌓아 하나의 횡단면처럼 회귀분석한다(2절). `linearmodels`의 `PooledOLS`로 추정한다. `linearmodels`는 회귀식을 문자열로 쓰지 않고, 종속변수 `y`와 설명변수 표 `X`를 따로 건넨다. 상수항은 저절로 들어가지 않으므로 `.assign(const=1)`로 1만 들어 있는 변수를 하나 더한다.

`.fit(cov_type='clustered', cluster_entity=True)`가 나라 단위 군집 표준오차다. 같은 나라의 관측을 한 묶음으로 보고 표준오차를 계산한다(2절의 "표준오차도 나라 단위로"). 같은 회귀식을 군집을 반영하지 않고(`.fit()`) 추정해 표준오차를 비교해 본다. 계수는 같고 표준오차만 다르다.

In [ ]:
y = p['로그수출']
X = p[['로그GDP', '로그거리', 'FTA']].assign(const=1)   # 상수항을 직접 더한다
CL = dict(cov_type='clustered', cluster_entity=True)      # 나라 단위 군집 표준오차

pool = PooledOLS(y, X).fit(**CL)
print(pd.DataFrame({'계수': pool.params, '표준오차': pool.std_errors, 'p값': pool.pvalues}).round(3))
print('R² =', round(pool.rsquared, 3), ' 관측 수', pool.nobs)

pool0 = PooledOLS(y, X).fit()                             # 군집을 반영하지 않은 표준오차
print('\nFTA 표준오차: 군집을 반영하지 않으면', round(pool0.std_errors['FTA'], 3),
      '/ 나라 단위 군집 표준오차', round(pool.std_errors['FTA'], 3))

## 5. 고정효과

### 자기 평균 빼기를 손으로

고정효과 모형은 나라마다 그 나라의 26년 평균을 빼고 회귀분석한 것이다(3절). 손으로 해 본다. `groupby('상대국코드')[v].transform('mean')`은 행마다 그 나라의 평균을 붙여 준다. 원래 값에서 이것을 빼면 "같은 나라의 변화"만 남는다. 평균을 뺀 변수끼리는 상수항이 필요 없으므로 회귀식 끝에 `- 1`을 붙여 상수항을 뺀다.

다음으로 나라 더미를 넣는 방법도 해 본다(3절의 "나라마다 자기 높이"). `C(상대국코드)`가 나라마다 스위치를 만든다. 나라가 206개이니 더미 205개(기준 하나를 뺀 것)가 들어간다. 두 방법의 FTA 계수는 똑같다.

In [ ]:
w = d.copy()
for v in ['로그수출', '로그GDP', 'FTA']:
    w[v + '_w'] = w[v] - w.groupby('상대국코드')[v].transform('mean')   # 자기 평균 빼기

m_hand = smf.ols('로그수출_w ~ 로그GDP_w + FTA_w - 1', data=w).fit()
m_dummy = smf.ols('로그수출 ~ 로그GDP + FTA + C(상대국코드)', data=d).fit()   # 나라 더미 205개
print('FTA 계수 — 자기 평균 빼기', round(m_hand.params['FTA_w'], 3), '/ 나라 더미', round(m_dummy.params['FTA'], 3))

### 고정효과 모형 — 표 3-1

실제로는 `PanelOLS`에 `entity_effects=True`(대상마다의 고정효과)를 주어 추정한다. 거리는 넣지 않는다. 나라마다 해가 바뀌어도 그대로라 자기 평균을 빼면 0이 되기 때문이다(3절의 "고정효과의 대가"). 넣으면 `linearmodels`가 오류를 낸다.

`rsquared_within`이 "같은 나라 안의 $R^2$"다. 자기 평균을 뺀 값, 즉 같은 나라의 변화 가운데 회귀식이 설명한 비율이다.

In [ ]:
fe = PanelOLS(y, p[['로그GDP', 'FTA']], entity_effects=True).fit(**CL)
print(pd.DataFrame({'계수': fe.params, '표준오차': fe.std_errors, 'p값': fe.pvalues}).round(3))
print('같은 나라 안의 R² =', round(fe.rsquared_within, 3), ' 관측 수', fe.nobs, ' 나라 수', int(fe.entity_info.total))
print('FTA → 퍼센트', round(np.expm1(fe.params['FTA']) * 100), '%')

### 군집 표준오차와 자유도

고정효과에서도 군집 표준오차가 판정을 가른다(3절의 "75%가 13%로"). 군집을 반영하지 않으면 FTA의 표준오차가 절반쯤으로 작아지고 $p$값은 0.01이 되어, 13%가 0과 분명히 구별되는 것처럼 보인다.

손으로 평균을 뺀 보통 OLS(위의 `m_hand`)는 계수는 같지만 표준오차가 조금 작게 나온다. 프로그램은 우리가 나라마다의 평균 206개를 먼저 계산해 썼다는 것을 모르고, 자유도를 4,909 − 2로 잡는다. 맞는 자유도는 그 206개까지 뺀 4,909 − 206 − 2다. 이 자료처럼 나라마다 관측이 스무 개가 넘으면 차이가 작지만, 대상마다 관측이 두세 번뿐이면 차이가 커진다. 그래서 직접 평균을 빼 보더라도 표준오차는 고정효과 추정 명령(`PanelOLS`)의 값을 쓴다.

표준오차가 군집을 반영하면 0.047에서 0.087로 커진다는 것을 정보의 양으로 읽어 볼 수도 있다. 표준오차는 관측 수의 제곱근에 반비례하므로, 표준오차가 1.85배가 되었다는 것은 정보로 치면 1.85의 제곱, 즉 약 3.4분의 1이라는 뜻이다. 한 나라를 평균 23.8해 관측했으니, 한 나라의 관측은 서로 독립된 관측 7개쯤의 정보를 가진 셈이다. 정확한 계산이 아니라 크기를 짐작해 보는 대략적인 계산이다.

In [ ]:
fe0 = PanelOLS(y, p[['로그GDP', 'FTA']], entity_effects=True).fit()   # 군집을 반영하지 않음
print('FTA 표준오차 — 군집 미반영', round(fe0.std_errors['FTA'], 4), '(p값', round(fe0.pvalues['FTA'], 3), ')',
      '/ 나라 단위 군집', round(fe.std_errors['FTA'], 3), '(p값', round(fe.pvalues['FTA'], 2), ')')
print('손으로 평균 뺀 OLS의 표준오차', round(m_hand.bse['FTA_w'], 4), '— 자유도를 잘못 잡아 조금 작다')

ratio = 0.087 / 0.047                         # 본문의 두 숫자
per = len(d) / d['상대국코드'].nunique()      # 나라마다 평균 몇 해를 관측했나
print('\n표준오차 비', round(ratio, 2), '→ 정보는', round(ratio ** 2, 1), '분의 1,',
      '나라마다 평균', round(per, 1), '해 → 독립된 관측으로 치면 약', round(per / ratio ** 2), '개')

### 두 차이를 나누는 계산

3절은 한국의 수출(로그)이 나라와 해에 따라 다른 정도 가운데 90%가 나라끼리의 차이이고 같은 나라의 변화는 10%뿐이라고 했다. 이 비율이 어떻게 나오는지 아이 두 명, 세 해의 작은 예로 먼저 계산한다. 지호의 키가 1학년부터 3학년까지 130, 136, 142cm이고, 서연이는 110, 116, 122cm라고 하자.

| | 뜻 |
|---|---|
| 전체 차이 | 키 − 여섯 값의 전체 평균 |
| 아이들끼리의 차이 | 그 아이의 자기 평균 − 전체 평균. 같은 아이라면 학년이 바뀌어도 같다 |
| 자신과의 차이 | 키 − 그 아이의 자기 평균. 같은 아이의 변화다 |

행마다 아이들끼리의 차이와 자신과의 차이를 더하면 전체 차이가 된다. 그냥 더하면 위아래로 벗어난 것이 상쇄되어 0이 되므로, 제곱해서 더한다(제곱합). 그러면 전체 차이의 제곱합(744)이 정확히 아이들끼리의 차이(600)와 자신과의 차이(144)로 나뉜다. 이 예에서는 81% 대 19%다. 두 아이는 키가 20cm나 차이 나는데 해마다 자라는 것은 6cm씩이기 때문이다.

같은 계산을 이 장의 자료 4,909개 관측에 하면 로그 수출은 90% 대 10%가 된다. 0 또는 1인 FTA는 54개국에서 어느 해에 0에서 1로 바뀌었으니 33% 대 67%로, 같은 나라의 변화가 더 크다(3절의 "고정효과의 대가", 4절의 "버린 정보가 아깝다").

In [ ]:
kids = pd.DataFrame({'아이': ['지호'] * 3 + ['서연'] * 3,
                     '키': [130, 136, 142, 110, 116, 122]})
kids['전체 차이'] = kids['키'] - kids['키'].mean()
kids['아이들끼리의 차이'] = kids.groupby('아이')['키'].transform('mean') - kids['키'].mean()
kids['자신과의 차이'] = kids['키'] - kids.groupby('아이')['키'].transform('mean')
print(kids)
ss = (kids[['전체 차이', '아이들끼리의 차이', '자신과의 차이']] ** 2).sum()   # 제곱합
print('\n제곱합', ss.to_dict())

def share_between(df, unit, v):
    # v가 대상과 해에 따라 다른 정도 가운데 대상끼리의 차이가 차지하는 비율
    total = ((df[v] - df[v].mean()) ** 2).sum()
    between = ((df.groupby(unit)[v].transform('mean') - df[v].mean()) ** 2).sum()
    return between / total

print('\n지호·서연: 아이들끼리의 차이', round(share_between(kids, '아이', '키') * 100), '%')
for v in ['로그수출', 'FTA']:
    b = share_between(d, '상대국코드', v)
    print(v, ': 나라끼리의 차이', round(b * 100), '%, 같은 나라의 변화', round((1 - b) * 100), '%')

### 그림으로 — 세 나라의 자기 평균 빼기

3절의 그림 3-2처럼 세 나라(말레이시아, 캐나다, 파키스탄)의 수출을 그대로 그린 것과 자기 평균을 뺀 것을 나란히 그린다. FTA가 발효된 해부터는 굵은 선으로 그린다. 자기 평균을 빼면 나라 사이의 높이 차이가 사라지고 세 선이 0 근처로 모인다.

In [ ]:
picks = {'MYS': '말레이시아', 'CAN': '캐나다', 'PAK': '파키스탄 (FTA 없음)'}
f3 = d[d['상대국코드'].isin(picks)].copy()
f3['평균 뺀 값'] = f3['로그수출'] - f3.groupby('상대국코드')['로그수출'].transform('mean')

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for code, name in picks.items():
    t = f3[f3['상대국코드'] == code].sort_values('연도')
    on = t[t['FTA'] == 1]
    for a, v in zip(ax, ['로그수출', '평균 뺀 값']):
        line, = a.plot(t['연도'], t[v], lw=1, label=name)
        a.plot(on['연도'], on[v], lw=3.5, color=line.get_color())   # FTA가 발효된 해부터 굵게
ax[0].set_title('그대로: 나라끼리의 차이가 크다')
ax[1].set_title('자기 평균을 빼면: 같은 나라의 변화만 남는다')
ax[1].axhline(0, color='gray', lw=0.8)
for a in ax:
    a.set_xlabel('연도 (굵은 선: FTA가 발효된 해부터)')
    a.legend()
plt.show()

## 6. 확률효과

### 확률효과 모형 — 표 4-1

`RandomEffects`로 추정한다. 고정효과와 달리 거리를 넣을 수 있다. 나라마다 자기 평균을 다 빼지 않고 $\theta$만큼만 빼므로, 남은 몫에 나라끼리의 차이가 살아 있기 때문이다(4절의 "거리는 어떻게 살아남았나").

`theta`에는 나라마다의 $\theta$가 들어 있다. 나라마다 관측한 해의 수가 달라 $\theta$도 조금씩 다르므로, 보고서는 평균을 적었다.

In [ ]:
re = RandomEffects(y, X).fit(**CL)
print(pd.DataFrame({'계수': re.params, '표준오차': re.std_errors, 'p값': re.pvalues}).round(3))
print('R² =', round(re.rsquared_overall, 3), ' 같은 나라 안의 R² =', round(re.rsquared_within, 3))
theta = re.theta.iloc[:, 0]                     # 나라마다의 θ
print('θ 평균', round(theta.mean(), 2), ' 가장 작은 θ', round(theta.min(), 2), ' 가장 큰 θ', round(theta.max(), 2))

### $\theta$를 손으로

$\theta$는 이렇게 정해진다(4절).

$$
\theta = 1 - \sqrt{\frac{\text{해마다의 흔들림의 분산}}{\text{관측한 해의 수} \times \text{자기 높이의 분산} + \text{해마다의 흔들림의 분산}}}
$$

두 분산은 확률효과가 추정해 `variance_decomposition`에 담아 둔다. `Effects`가 나라마다의 자기 높이 $a_i$의 분산, `Residual`이 해마다의 흔들림의 분산이다. 이 두 값을 넣어 관측한 해의 수에 따라 $\theta$를 계산하면 4절의 그림 4-4가 된다. 26년을 모두 관측한 나라는 0.91이고, 한 해만 관측한 나라는 0.57이다. 관측한 해가 많을수록 자기 평균을 믿을 만하니 많이 뺀다.

In [ ]:
re0 = RandomEffects(y, X).fit()                              # 분산 성분은 표준오차 계산법과 상관없다
s_u = re0.variance_decomposition['Effects']                  # 자기 높이의 분산
s_e = re0.variance_decomposition['Residual']                 # 해마다의 흔들림의 분산
print('자기 높이의 분산', round(s_u, 2), ' 해마다의 흔들림의 분산', round(s_e, 2),
      ' → 설명하지 못한 차이 가운데 나라마다의 몫', round(s_u / (s_u + s_e) * 100), '%')
for T in [1, 2, 5, 10, 26]:
    th = 1 - np.sqrt(s_e / (T * s_u + s_e))
    print(f'{T}년 관측: θ = {th:.2f}')

### 비교 방법에 따른 FTA의 계수 — 표 4-2

4절의 표 4-2는 나라끼리의 차이를 얼마나 쓰느냐에 따라 FTA의 계수가 어떻게 달라지는지 네 방법을 나란히 놓았다. 맨 위의 "나라 평균"은 나라마다 26년을 평균한 값 206개로 회귀분석한 것이다. 나라끼리의 차이만 쓴다(집단 간 추정량). 관측이 나라마다 하나이니 보통의 `smf.ols`로 하고, 표준오차는 HC1로 계산한다.

In [ ]:
avg = d.groupby('상대국코드')[['로그수출', '로그GDP', '로그거리', 'FTA']].mean()   # 나라마다 26년 평균
between = smf.ols('로그수출 ~ 로그GDP + 로그거리 + FTA', data=avg).fit(cov_type='HC1')

rows = {'나라 평균 (나라끼리의 차이만)': (between.params['FTA'], between.bse['FTA']),
        '합동 OLS (그대로 섞어)': (pool.params['FTA'], pool.std_errors['FTA']),
        '확률효과 (θ만큼 섞어)': (re.params['FTA'], re.std_errors['FTA']),
        '고정효과 (같은 나라의 변화만)': (fe.params['FTA'], fe.std_errors['FTA'])}
print(pd.DataFrame(rows, index=['FTA 계수', '표준오차']).T.round(3))

## 7. 하우스만 검정

### 통계량을 손으로 — FTA 하나로

하우스만 통계량은 두 모형의 계수 추정값의 차이를, 그 차이가 우연히 생길 수 있는 정도와 비교한다(5절).

$$
\text{하우스만 통계량} \approx \frac{(\text{고정효과 계수} - \text{확률효과 계수})^2}{\text{고정효과 계수의 분산} - \text{확률효과 계수의 분산}}
$$

주의할 점이 둘 있다. 첫째, 여기에는 군집을 반영하지 않은 표준오차를 쓴다. 이 식은 "확률효과의 가정이 맞으면 확률효과가 가장 정밀하다"는 성질에 기대는데, 이 성질은 군집을 반영하지 않은 표준오차에서 성립하기 때문이다. 보고서 표의 군집 표준오차(0.087, 0.086)를 넣으면 0.2쯤이 나와 엉뚱한 결론에 이른다. 둘째, 분모는 거의 같은 두 수의 차이라서 아주 작고, 반올림에 민감하다. 계수와 표준오차를 소수 셋째 자리(0.121과 0.128, 0.048과 0.047)로 반올림해 넣으면 통계량이 0.5쯤으로 줄어든다. 그래서 계산은 반올림하지 않은 값으로 한다.

In [ ]:
b_fe, b_re = fe0.params['FTA'], re0.params['FTA']              # 군집을 반영하지 않은 판
se_fe, se_re = fe0.std_errors['FTA'], re0.std_errors['FTA']
H1 = (b_fe - b_re) ** 2 / (se_fe ** 2 - se_re ** 2)
print(f'고정효과 {b_fe:.5f} (표준오차 {se_fe:.5f}), 확률효과 {b_re:.5f} (표준오차 {se_re:.5f})')
print('FTA 하나로 계산한 하우스만 통계량', round(H1, 2))

H_round = (0.121 - 0.128) ** 2 / (0.048 ** 2 - 0.047 ** 2)   # 반올림한 값을 넣으면
print('소수 셋째 자리로 반올림해 넣으면', round(H_round, 2))

### 두 계수를 함께 — 보고서의 1.51

보고서의 하우스만 검정은 GDP와 FTA 두 계수를 함께 넣었다. 계수가 여러 개면 차이를 벡터로, 분산을 행렬로 모아 같은 계산을 한다. `@`는 행렬 곱이고, `np.linalg.pinv`는 역행렬이다. 통계량은 자유도가 계수의 수(2)인 카이제곱 분포를 따르므로, `chi2.sf`로 $p$값을 구한다.

같은 함수를 8절에서도 쓴다.

In [ ]:
def hausman(f, r, names=('로그GDP', 'FTA')):
    names = list(names)
    diff = f.params[names] - r.params[names]                    # 계수의 차이
    V = f.cov.loc[names, names] - r.cov.loc[names, names]       # 분산의 차이(행렬)
    H = float(diff @ np.linalg.pinv(V) @ diff)
    return H, chi2.sf(H, len(names))

H, pval = hausman(fe0, re0)
print(f'하우스만 통계량 {H:.2f}, p값 {pval:.2f}')

## 8. 연도 효과

연도마다 더미를 하나씩 달아 해마다 모든 나라에 함께 일어난 변화를 따로 잡는다(5절의 "모두 함께 자라는 몫"). `pd.get_dummies`로 연도 더미를 만들고, 기준 해(1995년) 하나는 뺀다(`drop_first=True`). 만든 더미를 `join`으로 설명변수 표에 붙인다.

고정효과와 확률효과에 모두 연도 효과를 넣고, 하우스만 검정을 다시 한다. 이 장의 자료에서는 연도 효과를 넣자 하우스만 검정이 기각되었다. GDP의 계수가 두 모형에서 벌어졌기 때문이다.

In [ ]:
yd = pd.get_dummies(p.index.get_level_values('연도'), prefix='연도', drop_first=True, dtype=float)
yd.index = p.index                                             # 패널의 꼬리표를 맞춘다

fe_y = PanelOLS(y, p[['로그GDP', 'FTA']].join(yd), entity_effects=True).fit(**CL)
re_y = RandomEffects(y, X.join(yd)).fit(**CL)

table = pd.DataFrame({'고정효과': fe.params, '확률효과': re.params,
                      '고정효과 + 연도 효과': fe_y.params, '확률효과 + 연도 효과': re_y.params})
print(table.loc[['로그GDP', '로그거리', 'FTA']].round(3).fillna('—'))   # 고정효과에는 거리가 없다
print('FTA p값 — 고정효과 + 연도 효과', round(fe_y.pvalues['FTA'], 2), ', 확률효과 + 연도 효과', round(re_y.pvalues['FTA'], 2))

# 하우스만 검정은 군집을 반영하지 않은 판으로
fe_y0 = PanelOLS(y, p[['로그GDP', 'FTA']].join(yd), entity_effects=True).fit()
re_y0 = RandomEffects(y, X.join(yd)).fit()
H, pval = hausman(fe_y0, re_y0)
print(f'\n연도 효과를 넣은 하우스만 통계량 {H:.2f}, p값 {pval:.3f}')

### 해마다의 연도 효과

본문의 표 5-1은 본문에서 읽는 해만 실었다. 여기서는 25개를 모두 본다. 연도 효과는 1995년에 비해 그해 모든 나라로의 수출이 함께 얼마나 높았는지를 로그로 나타낸 것이다. 2008년에서 2009년으로 0.168 떨어진 것은, 퍼센트로 바꾸면 약 15%가 함께 줄었다는 뜻이다. 세계 금융위기의 해다.

In [ ]:
ye = fe_y.params[yd.columns]                                   # 연도 효과 25개
ye.index = [int(c.split('_')[1]) for c in ye.index]           # '연도_2009' → 2009
print(ye.round(3).to_string())
print('2008 → 2009:', round(np.expm1(ye[2009] - ye[2008]) * 100), '%,  2019 → 2020:', round(np.expm1(ye[2020] - ye[2019]) * 100), '%')

plt.figure(figsize=(7, 3.5))
plt.plot(ye.index, ye.values, marker='o')
plt.axhline(0, color='gray', lw=0.8)
plt.title('연도 효과 (고정효과 + 연도 효과, 1995년 = 0)')
plt.xlabel('연도')
plt.ylabel('로그 수출')
plt.show()

## 9. 더 해 보기: 발효 뒤 몇 해째인가

본문에서는 다루지 않은 탐색이다. FTA 더미 하나는 발효 뒤 모든 해의 평균이다. 관세를 여러 해에 걸쳐 내리는 FTA가 많으니 효과가 해가 지나며 커질 수 있는데, 그것을 보려면 발효 뒤 몇 해째인지에 따라 더미를 나눠 넣는다. 나라마다 FTA가 처음 1이 된 해(2절의 `start`)를 기준으로 몇 해째인지(`since`)를 계산하고, 0~2년, 3~5년, 6년 이후의 더미 셋을 만든다. FTA를 맺지 않은 나라는 `since`가 비어 있어(NaN) 세 더미가 모두 0이 된다.

연도 효과를 넣기 전과 넣은 뒤를 나란히 추정한다. 구간마다 들어 있는 나라가 다르다는 점도 함께 센다. 자료가 2020년에서 끝나니, 6년 이후 구간에는 2014년까지 FTA를 맺은 나라만 들어 있다. 결과는 모두 0과 분명히 구별되지 않고, FTA를 맺은 해가 나라마다 달라 이런 계수는 한쪽으로 치우칠 수 있으니 모양을 짐작하는 정도로만 본다.

In [ ]:
since = d['연도'] - d['상대국코드'].map(start)                 # 발효 뒤 몇 해째인가 (FTA가 없는 나라는 NaN)
bins = {'발효 0~2년': (0, 2), '발효 3~5년': (3, 5), '발효 6년 이후': (6, 99)}
for name, (lo, hi) in bins.items():
    d[name] = since.between(lo, hi).astype(int)
pb = d.set_index(['상대국코드', '연도']).sort_index()
B = list(bins)

ev0 = PanelOLS(y, pb[['로그GDP'] + B], entity_effects=True).fit(**CL)
ev1 = PanelOLS(y, pb[['로그GDP'] + B].join(yd), entity_effects=True).fit(**CL)
out = pd.DataFrame({'고정효과': ev0.params[B], '(표준오차)': ev0.std_errors[B],
                    '고정효과 + 연도 효과': ev1.params[B], '(표준오차) ': ev1.std_errors[B],
                    '들어 있는 나라': [pb[pb[b] == 1].index.get_level_values(0).nunique() for b in B]})
print(out.round(3))

## 정리

이 노트북의 코드는 이 장에서 본 숫자를 직접 만들어 냈다. 보고서의 네 모형은 3절의 `smf.ols`와 4~6절의 `PooledOLS`, `PanelOLS`, `RandomEffects`에서 나오고, 본문의 표는 그 모형에 연도 효과나 구간 더미를 더하거나(8, 9절) 같은 회귀식을 다른 해에 적용한(3절) 것이다. 자기 평균 빼기, $\theta$, 하우스만 통계량은 손으로 계산해도 프로그램과 같은 값이 나온다(5, 6, 7절).

자기 자료로 분석하려면 1절의 `CSV` 파일 이름을 바꾸고 차례로 실행한다. 3절의 `YEAR`는 한 해 비교에 쓸 해로 바꾼다. 3절의 2003년과 9절의 발효 뒤 구간은 한국의 FTA 연도에 맞춘 것이라, 다른 수출국의 자료에서는 첫 FTA보다 앞선 해로 바꿔야 한다. 5절의 그림도 세 나라의 코드를 그 자료에 있는 나라로 바꿔야 한다. AI가 만든 보고서를 받았다면, 이 코드로 같은 숫자가 나오는지 확인해 보는 것이 가장 확실한 점검이다.